# 08 - Análisis de Stock: CatBoost vs Naive

Este notebook implementa el análisis de costes de inventario comparando:
- **Modelo CatBoost**: Predicciones del mejor modelo ML
- **Modelo Naive**: Media móvil de 7 días

Se utiliza la fórmula de Yamazaki (2015) para calcular el stock de seguridad:
$$SS = Z \times \sigma \times \sqrt{L + 1}$$

Donde:
- $Z$: Factor de servicio (1.645 para 95%, 2.326 para 99%)
- $\sigma$: Desviación estándar del error (RMSE)
- $L$: Lead time (días de aprovisionamiento)

**Referencia**: Yamazaki, Y. (2015). DOI: https://doi.org/10.1080/00207543.2015.1076179

## 1. Configuración e Importaciones

In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Detectar si estamos en Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Configurar el directorio de trabajo según el entorno
if IN_COLAB:
    os.chdir('TFMDS')
else:
    # Detectar si estamos en Codespaces o VS Code local
    if os.path.exists('/workspaces/TFMDS'):
        # Entorno Codespaces
        os.chdir('/workspaces/TFMDS')
    else:
        # En VS Code local, nos movemos al directorio raíz del proyecto
        # Usa raw string para evitar errores de escape en rutas Windows
        current_dir = r'C:\Users\jmora\Documents\TFMDS'
        os.chdir(current_dir)

# OPCIONAL: Para verificar que estás en la ruta correcta y ver las carpetas
print("Directorio de trabajo actual:", os.getcwd())

# Parámetros de costes (ajustables según negocio)
TASA_ALMACENAMIENTO_ANUAL = 0.20  # 20% del valor del producto
COSTE_RUPTURA_UNITARIO = 0.10     # 10% del precio como penalización
NIVEL_SERVICIO = 0.95             # 95% de nivel de servicio
Z_SCORE = 1.645                   # Factor Z para 95% de confianza

print("Configuración cargada")
print(f"Nivel de servicio: {NIVEL_SERVICIO*100}%")
print(f"Z-score: {Z_SCORE}")
print(f"Tasa almacenamiento anual: {TASA_ALMACENAMIENTO_ANUAL*100}%")
print(f"Coste ruptura unitario: {COSTE_RUPTURA_UNITARIO*100}%")

Directorio de trabajo actual: /workspaces/TFMDS
Configuración cargada
Nivel de servicio: 95.0%
Z-score: 1.645
Tasa almacenamiento anual: 20.0%
Coste ruptura unitario: 10.0%


## 2. Carga de Datos

In [ ]:
# Cargar predicciones de CatBoost
df_test = pd.read_csv('../datos/df_test_catboost.csv', sep=';', decimal=',')
print(f"Datos de test cargados: {df_test.shape}")
print(f"Columnas: {df_test.columns.tolist()}")

# Cargar datos de ciclo de aprovisionamiento
df_ciclos = pd.read_csv('../datos/DatosCicloAprovisionamiento.csv', sep=';', decimal=',')
print(f"\nDatos de ciclos: {df_ciclos.shape}")
print(f"Columnas: {df_ciclos.columns.tolist()}")

# Cargar precios medios
df_precios = pd.read_csv('../datos/DatosPrecioMedio.csv', sep=';', decimal=',')
# Convertir precio de formato europeo (coma) a float
if df_precios['eurPrecioMedio'].dtype == 'object':
    df_precios['eurPrecioMedio'] = df_precios['eurPrecioMedio'].str.replace(',', '.').astype(float)
print(f"\nDatos de precios: {df_precios.shape}")
print(f"Columnas: {df_precios.columns.tolist()}")

# Vista previa
print("\nPrimeras filas de df_test:")
display(df_test.head())
print("\nPrimeras filas de df_ciclos:")
display(df_ciclos.head())
print("\nPrimeras filas de df_precios:")
display(df_precios.head())

## 3. Cálculo de RMSE para CatBoost

In [ ]:
# Calcular RMSE por producto para CatBoost
rmse_catboost = df_test.groupby('producto').apply(
    lambda x: np.sqrt(mean_squared_error(x['udsVenta'], x['udsVentaPred']))
).reset_index(name='rmse_catboost')

print(f"RMSE calculado para {len(rmse_catboost)} productos")
print(f"\nEstadísticas de RMSE CatBoost:")
print(rmse_catboost['rmse_catboost'].describe())
print(f"\nPrimeros productos:")
display(rmse_catboost.head(10))

## 4. Cálculo de RMSE para Modelo Naive (Media Móvil 7 días)

In [ ]:
# Ordenar por producto y secuencia para rolling window
df_test_sorted = df_test.sort_values(['producto', 'idSecuencia']).copy()

# Calcular predicción naive: media móvil de 7 días desplazada 1 período
df_test_sorted['udsVentaPred_naive'] = df_test_sorted.groupby('producto')['udsVenta'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean().shift(1)
)

# Para el primer valor de cada producto, usar la media global del producto
df_test_sorted['udsVentaPred_naive'] = df_test_sorted.groupby('producto')['udsVentaPred_naive'].transform(
    lambda x: x.fillna(df_test_sorted.loc[x.index, 'udsVenta'].mean())
)

# Calcular RMSE por producto para Naive
rmse_naive = df_test_sorted.groupby('producto').apply(
    lambda x: np.sqrt(mean_squared_error(x['udsVenta'], x['udsVentaPred_naive']))
).reset_index(name='rmse_naive')

print(f"RMSE Naive calculado para {len(rmse_naive)} productos")
print(f"\nEstadísticas de RMSE Naive:")
print(rmse_naive['rmse_naive'].describe())
print(f"\nPrimeros productos:")
display(rmse_naive.head(10))

## 5. Cálculo de Stock de Seguridad (Fórmula de Yamazaki)

In [ ]:
# Fusionar todos los datos
df_analisis = rmse_catboost.merge(rmse_naive, on='producto')
df_analisis = df_analisis.merge(df_ciclos, on='producto')
df_analisis = df_analisis.merge(df_precios, on='producto')

print(f"Datos fusionados: {df_analisis.shape}")

# Calcular stock de seguridad según fórmula de Yamazaki
# SS = Z × σ × √(L + 1)
df_analisis['ss_catboost'] = Z_SCORE * df_analisis['rmse_catboost'] * np.sqrt(df_analisis['diasLeadtime'] + 1)
df_analisis['ss_naive'] = Z_SCORE * df_analisis['rmse_naive'] * np.sqrt(df_analisis['diasLeadtime'] + 1)

print("\nStock de seguridad calculado")
print(f"\nEstadísticas SS CatBoost:")
print(df_analisis['ss_catboost'].describe())
print(f"\nEstadísticas SS Naive:")
print(df_analisis['ss_naive'].describe())

# Diferencia en stock de seguridad
df_analisis['ss_diferencia'] = df_analisis['ss_naive'] - df_analisis['ss_catboost']
df_analisis['ss_reduccion_pct'] = (df_analisis['ss_diferencia'] / df_analisis['ss_naive']) * 100

print(f"\nReducción media en SS: {df_analisis['ss_reduccion_pct'].mean():.2f}%")
display(df_analisis[['producto', 'rmse_catboost', 'rmse_naive', 'ss_catboost', 'ss_naive', 'ss_reduccion_pct']].head(10))

## 6. Cálculo de Stock Medio y Stock Máximo

In [ ]:
# Calcular demanda media por producto
demanda_media = df_test.groupby('producto')['udsVenta'].mean().reset_index(name='demanda_media_diaria')
df_analisis = df_analisis.merge(demanda_media, on='producto')

# Stock de ciclo = (Demanda media × Días entre pedidos) / 2
df_analisis['stock_ciclo'] = (df_analisis['demanda_media_diaria'] * df_analisis['diasEntrePedidos']) / 2

# Stock medio = Stock de ciclo + Stock de seguridad
df_analisis['stock_medio_catboost'] = df_analisis['stock_ciclo'] + df_analisis['ss_catboost']
df_analisis['stock_medio_naive'] = df_analisis['stock_ciclo'] + df_analisis['ss_naive']

# Stock máximo = Demanda media × (Leadtime + Días entre pedidos) + Stock de seguridad
df_analisis['stock_max_catboost'] = df_analisis['demanda_media_diaria'] * (
    df_analisis['diasLeadtime'] + df_analisis['diasEntrePedidos']
) + df_analisis['ss_catboost']

df_analisis['stock_max_naive'] = df_analisis['demanda_media_diaria'] * (
    df_analisis['diasLeadtime'] + df_analisis['diasEntrePedidos']
) + df_analisis['ss_naive']

print("Stock medio y máximo calculados")
print(f"\nStock medio CatBoost: {df_analisis['stock_medio_catboost'].sum():.0f} unidades")
print(f"Stock medio Naive: {df_analisis['stock_medio_naive'].sum():.0f} unidades")
print(f"\nReducción total en stock medio: {(df_analisis['stock_medio_naive'].sum() - df_analisis['stock_medio_catboost'].sum()):.0f} unidades")

display(df_analisis[['producto', 'stock_ciclo', 'stock_medio_catboost', 'stock_medio_naive', 'stock_max_catboost', 'stock_max_naive']].head(10))

## 7. Cálculo de Costes

In [ ]:
# Coste de almacenamiento anual = Stock medio × Precio × Tasa de almacenamiento
df_analisis['coste_almacen_catboost'] = df_analisis['stock_medio_catboost'] * df_analisis['eurPrecioMedio'] * TASA_ALMACENAMIENTO_ANUAL
df_analisis['coste_almacen_naive'] = df_analisis['stock_medio_naive'] * df_analisis['eurPrecioMedio'] * TASA_ALMACENAMIENTO_ANUAL

# Estimar roturas de stock basadas en nivel de servicio no cubierto
# Asumimos que el error no cubierto genera rupturas proporcionales
dias_test = df_test.groupby('producto')['idSecuencia'].nunique().reset_index(name='dias_observados')
df_analisis = df_analisis.merge(dias_test, on='producto')

# Coste de ruptura = RMSE × Días × Precio × Coste ruptura unitario
df_analisis['coste_ruptura_catboost'] = df_analisis['rmse_catboost'] * df_analisis['dias_observados'] * df_analisis['eurPrecioMedio'] * COSTE_RUPTURA_UNITARIO
df_analisis['coste_ruptura_naive'] = df_analisis['rmse_naive'] * df_analisis['dias_observados'] * df_analisis['eurPrecioMedio'] * COSTE_RUPTURA_UNITARIO

# Coste total
df_analisis['coste_total_catboost'] = df_analisis['coste_almacen_catboost'] + df_analisis['coste_ruptura_catboost']
df_analisis['coste_total_naive'] = df_analisis['coste_almacen_naive'] + df_analisis['coste_ruptura_naive']

# Ahorro
df_analisis['ahorro_total'] = df_analisis['coste_total_naive'] - df_analisis['coste_total_catboost']
df_analisis['ahorro_pct'] = (df_analisis['ahorro_total'] / df_analisis['coste_total_naive']) * 100

print("="*80)
print("RESUMEN DE COSTES ANUALES")
print("="*80)
print(f"\n{'MODELO CATBOOST':^40}")
print("-"*80)
print(f"  Coste almacenamiento: {df_analisis['coste_almacen_catboost'].sum():>15,.2f} €")
print(f"  Coste ruptura stock:  {df_analisis['coste_ruptura_catboost'].sum():>15,.2f} €")
print(f"  COSTE TOTAL:          {df_analisis['coste_total_catboost'].sum():>15,.2f} €")

print(f"\n{'MODELO NAIVE':^40}")
print("-"*80)
print(f"  Coste almacenamiento: {df_analisis['coste_almacen_naive'].sum():>15,.2f} €")
print(f"  Coste ruptura stock:  {df_analisis['coste_ruptura_naive'].sum():>15,.2f} €")
print(f"  COSTE TOTAL:          {df_analisis['coste_total_naive'].sum():>15,.2f} €")

print(f"\n{'AHORRO CON CATBOOST':^40}")
print("="*80)
print(f"  Ahorro total:         {df_analisis['ahorro_total'].sum():>15,.2f} €")
print(f"  Reducción:            {(df_analisis['ahorro_total'].sum() / df_analisis['coste_total_naive'].sum() * 100):>15,.2f} %")
print("="*80)

display(df_analisis[['producto', 'coste_total_catboost', 'coste_total_naive', 'ahorro_total', 'ahorro_pct']].head(10))

## 8. Exportar Resultados

In [ ]:
# Seleccionar columnas relevantes para exportar
columnas_exportar = [
    'producto',
    'eurPrecioMedio',
    'diasEntrePedidos',
    'diasLeadtime',
    'demanda_media_diaria',
    'rmse_catboost',
    'rmse_naive',
    'ss_catboost',
    'ss_naive',
    'ss_reduccion_pct',
    'stock_medio_catboost',
    'stock_medio_naive',
    'stock_max_catboost',
    'stock_max_naive',
    'coste_almacen_catboost',
    'coste_almacen_naive',
    'coste_ruptura_catboost',
    'coste_ruptura_naive',
    'coste_total_catboost',
    'coste_total_naive',
    'ahorro_total',
    'ahorro_pct'
]

df_exportar = df_analisis[columnas_exportar].copy()

# Ordenar por ahorro total descendente
df_exportar = df_exportar.sort_values('ahorro_total', ascending=False)

# Exportar a CSV
ruta_salida = '../datos/analisis_stock_comparativo.csv'
df_exportar.to_csv(ruta_salida, sep=';', decimal=',', index=False)

print(f"Resultados exportados a: {ruta_salida}")
print(f"Total de productos analizados: {len(df_exportar)}")
print(f"\nProductos con mayor ahorro:")
display(df_exportar[['producto', 'ahorro_total', 'ahorro_pct']].head(10))

## 9. Resumen Ejecutivo

In [ ]:
# Crear resumen ejecutivo
print("="*80)
print("RESUMEN EJECUTIVO: ANÁLISIS DE INVENTARIO")
print("="*80)
print(f"\nModelos comparados:")
print(f"  - CatBoost (modelo ML optimizado)")
print(f"  - Naive (media móvil 7 días)")
print(f"\nParámetros:")
print(f"  - Nivel de servicio: {NIVEL_SERVICIO*100}%")
print(f"  - Z-score: {Z_SCORE}")
print(f"  - Tasa almacenamiento: {TASA_ALMACENAMIENTO_ANUAL*100}% anual")
print(f"  - Coste ruptura: {COSTE_RUPTURA_UNITARIO*100}% del precio")

print(f"\nProductos analizados: {len(df_analisis)}")
print(f"\nMejora en precisión (RMSE):")
rmse_mejora = ((df_analisis['rmse_naive'].mean() - df_analisis['rmse_catboost'].mean()) / df_analisis['rmse_naive'].mean()) * 100
print(f"  - RMSE medio Naive:     {df_analisis['rmse_naive'].mean():.2f} unidades")
print(f"  - RMSE medio CatBoost:  {df_analisis['rmse_catboost'].mean():.2f} unidades")
print(f"  - Mejora:               {rmse_mejora:.2f}%")

print(f"\nReducción en stock de seguridad:")
ss_total_naive = df_analisis['ss_naive'].sum()
ss_total_catboost = df_analisis['ss_catboost'].sum()
ss_reduccion = ((ss_total_naive - ss_total_catboost) / ss_total_naive) * 100
print(f"  - SS total Naive:       {ss_total_naive:,.0f} unidades")
print(f"  - SS total CatBoost:    {ss_total_catboost:,.0f} unidades")
print(f"  - Reducción:            {ss_reduccion:.2f}%")

print(f"\nReducción en stock medio:")
stock_medio_naive = df_analisis['stock_medio_naive'].sum()
stock_medio_catboost = df_analisis['stock_medio_catboost'].sum()
stock_reduccion = ((stock_medio_naive - stock_medio_catboost) / stock_medio_naive) * 100
print(f"  - Stock medio Naive:    {stock_medio_naive:,.0f} unidades")
print(f"  - Stock medio CatBoost: {stock_medio_catboost:,.0f} unidades")
print(f"  - Reducción:            {stock_reduccion:.2f}%")

print(f"\nImpacto económico anual:")
coste_total_naive = df_analisis['coste_total_naive'].sum()
coste_total_catboost = df_analisis['coste_total_catboost'].sum()
ahorro_total = df_analisis['ahorro_total'].sum()
ahorro_pct = (ahorro_total / coste_total_naive) * 100
print(f"  - Coste total Naive:    {coste_total_naive:>15,.2f} €")
print(f"  - Coste total CatBoost: {coste_total_catboost:>15,.2f} €")
print(f"  - AHORRO ANUAL:         {ahorro_total:>15,.2f} €")
print(f"  - Reducción de coste:   {ahorro_pct:>15,.2f} %")

# Productos con mayor impacto
top_10_ahorro = df_analisis.nlargest(10, 'ahorro_total')
ahorro_top10 = top_10_ahorro['ahorro_total'].sum()
print(f"\nTop 10 productos representan:")
print(f"  - Ahorro: {ahorro_top10:,.2f} € ({(ahorro_top10/ahorro_total)*100:.1f}% del total)")

print(f"\n" + "="*80)
print(f"CONCLUSIÓN: El modelo CatBoost reduce los costes de inventario en {ahorro_pct:.2f}%")
print(f"generando un ahorro anual de {ahorro_total:,.2f} € mediante predicciones más precisas.")
print("="*80)

## 10. Visualizaciones Comparativas

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de estilo
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 11

print("Configuración de gráficos lista")

### 10.1. Comparativa de RMSE por Producto (Top 20)

In [ ]:
# Seleccionar top 20 productos por volumen de venta
top_20_productos = df_analisis.nlargest(20, 'demanda_media_diaria')

fig, ax = plt.subplots(figsize=(16, 8))

x = np.arange(len(top_20_productos))
width = 0.35

bars1 = ax.bar(x - width/2, top_20_productos['rmse_naive'], width, 
               label='Naive', color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x + width/2, top_20_productos['rmse_catboost'], width, 
               label='CatBoost', color='#2ecc71', alpha=0.8)

ax.set_xlabel('Producto', fontsize=13, fontweight='bold')
ax.set_ylabel('RMSE (unidades)', fontsize=13, fontweight='bold')
ax.set_title('Comparativa de Error de Predicción (RMSE) - Top 20 Productos por Demanda', 
             fontsize=15, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(top_20_productos['producto'], rotation=45, ha='right')
ax.legend(fontsize=12, loc='upper left')
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Añadir valores sobre las barras
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('../datos/grafico_rmse_comparativa.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Mejora media en RMSE (Top 20): {((top_20_productos['rmse_naive'].mean() - top_20_productos['rmse_catboost'].mean()) / top_20_productos['rmse_naive'].mean() * 100):.2f}%")

### 10.2. Comparativa de Stock de Seguridad

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Análisis de Stock de Seguridad: CatBoost vs Naive', 
             fontsize=16, fontweight='bold', y=0.995)

# 1. Distribución de stock de seguridad
ax1 = axes[0, 0]
ax1.hist(df_analisis['ss_naive'], bins=50, alpha=0.6, label='Naive', color='#e74c3c', edgecolor='black')
ax1.hist(df_analisis['ss_catboost'], bins=50, alpha=0.6, label='CatBoost', color='#2ecc71', edgecolor='black')
ax1.axvline(df_analisis['ss_naive'].mean(), color='#e74c3c', linestyle='--', linewidth=2, label=f'Media Naive: {df_analisis["ss_naive"].mean():.1f}')
ax1.axvline(df_analisis['ss_catboost'].mean(), color='#2ecc71', linestyle='--', linewidth=2, label=f'Media CatBoost: {df_analisis["ss_catboost"].mean():.1f}')
ax1.set_xlabel('Stock de Seguridad (unidades)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Frecuencia', fontsize=12, fontweight='bold')
ax1.set_title('Distribución de Stock de Seguridad', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3, linestyle='--')

# 2. Reducción porcentual por producto
ax2 = axes[0, 1]
colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in df_analisis['ss_reduccion_pct']]
ax2.scatter(df_analisis['demanda_media_diaria'], df_analisis['ss_reduccion_pct'], 
            alpha=0.6, s=50, c=colors, edgecolors='black', linewidth=0.5)
ax2.axhline(0, color='black', linestyle='-', linewidth=1)
ax2.axhline(df_analisis['ss_reduccion_pct'].mean(), color='blue', linestyle='--', 
            linewidth=2, label=f'Media: {df_analisis["ss_reduccion_pct"].mean():.1f}%')
ax2.set_xlabel('Demanda Media Diaria (unidades)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Reducción en SS (%)', fontsize=12, fontweight='bold')
ax2.set_title('Reducción de Stock de Seguridad vs Demanda', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3, linestyle='--')

# 3. Top 15 productos con mayor reducción en unidades
ax3 = axes[1, 0]
top_15_reduccion = df_analisis.nlargest(15, 'ss_diferencia')
y_pos = np.arange(len(top_15_reduccion))
bars = ax3.barh(y_pos, top_15_reduccion['ss_diferencia'], color='#3498db', alpha=0.8, edgecolor='black')
ax3.set_yticks(y_pos)
ax3.set_yticklabels(top_15_reduccion['producto'], fontsize=10)
ax3.set_xlabel('Reducción en SS (unidades)', fontsize=12, fontweight='bold')
ax3.set_title('Top 15 Productos con Mayor Reducción en SS', fontsize=13, fontweight='bold')
ax3.grid(axis='x', alpha=0.3, linestyle='--')

# Añadir valores
for i, (bar, val) in enumerate(zip(bars, top_15_reduccion['ss_diferencia'])):
    ax3.text(val + 0.5, i, f'{val:.1f}', va='center', fontsize=9)

# 4. Relación entre Lead Time y reducción de SS
ax4 = axes[1, 1]
scatter = ax4.scatter(df_analisis['diasLeadtime'], df_analisis['ss_diferencia'],
                      c=df_analisis['ss_reduccion_pct'], cmap='RdYlGn', 
                      s=80, alpha=0.7, edgecolors='black', linewidth=0.5)
ax4.set_xlabel('Lead Time (días)', fontsize=12, fontweight='bold')
ax4.set_ylabel('Reducción en SS (unidades)', fontsize=12, fontweight='bold')
ax4.set_title('Impacto del Lead Time en la Reducción de SS', fontsize=13, fontweight='bold')
ax4.grid(alpha=0.3, linestyle='--')
cbar = plt.colorbar(scatter, ax=ax4)
cbar.set_label('Reducción SS (%)', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../datos/grafico_stock_seguridad.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Reducción total en SS: {df_analisis['ss_diferencia'].sum():,.0f} unidades ({df_analisis['ss_reduccion_pct'].mean():.2f}% promedio)")

### 10.3. Análisis de Costes

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Análisis Comparativo de Costes: CatBoost vs Naive', 
             fontsize=16, fontweight='bold', y=0.995)

# 1. Comparativa de costes totales por tipo
ax1 = axes[0, 0]
categorias = ['Almacenamiento', 'Ruptura', 'Total']
costes_naive = [
    df_analisis['coste_almacen_naive'].sum(),
    df_analisis['coste_ruptura_naive'].sum(),
    df_analisis['coste_total_naive'].sum()
]
costes_catboost = [
    df_analisis['coste_almacen_catboost'].sum(),
    df_analisis['coste_ruptura_catboost'].sum(),
    df_analisis['coste_total_catboost'].sum()
]

x = np.arange(len(categorias))
width = 0.35

bars1 = ax1.bar(x - width/2, costes_naive, width, label='Naive', color='#e74c3c', alpha=0.8, edgecolor='black')
bars2 = ax1.bar(x + width/2, costes_catboost, width, label='CatBoost', color='#2ecc71', alpha=0.8, edgecolor='black')

ax1.set_ylabel('Coste Anual (€)', fontsize=12, fontweight='bold')
ax1.set_title('Comparativa de Costes Anuales por Categoría', fontsize=13, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(categorias, fontsize=11)
ax1.legend(fontsize=11)
ax1.grid(axis='y', alpha=0.3, linestyle='--')

# Añadir valores sobre barras
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:,.0f}€', ha='center', va='bottom', fontsize=9, fontweight='bold')

# 2. Top 15 productos con mayor ahorro
ax2 = axes[0, 1]
top_15_ahorro = df_analisis.nlargest(15, 'ahorro_total')
y_pos = np.arange(len(top_15_ahorro))
bars = ax2.barh(y_pos, top_15_ahorro['ahorro_total'], color='#27ae60', alpha=0.8, edgecolor='black')
ax2.set_yticks(y_pos)
ax2.set_yticklabels(top_15_ahorro['producto'], fontsize=10)
ax2.set_xlabel('Ahorro Anual (€)', fontsize=12, fontweight='bold')
ax2.set_title('Top 15 Productos con Mayor Ahorro', fontsize=13, fontweight='bold')
ax2.grid(axis='x', alpha=0.3, linestyle='--')

# Añadir valores
for i, (bar, val) in enumerate(zip(bars, top_15_ahorro['ahorro_total'])):
    ax2.text(val + 10, i, f'{val:,.0f}€', va='center', fontsize=9)

# 3. Distribución de ahorro porcentual
ax3 = axes[1, 0]
n, bins, patches = ax3.hist(df_analisis['ahorro_pct'], bins=40, color='#3498db', 
                             alpha=0.7, edgecolor='black', linewidth=1)
ax3.axvline(df_analisis['ahorro_pct'].mean(), color='red', linestyle='--', 
            linewidth=2, label=f'Media: {df_analisis["ahorro_pct"].mean():.2f}%')
ax3.axvline(df_analisis['ahorro_pct'].median(), color='orange', linestyle='--', 
            linewidth=2, label=f'Mediana: {df_analisis["ahorro_pct"].median():.2f}%')
ax3.set_xlabel('Ahorro (%)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Frecuencia', fontsize=12, fontweight='bold')
ax3.set_title('Distribución del Ahorro Porcentual por Producto', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(alpha=0.3, linestyle='--')

# 4. Diagrama de Pareto del ahorro acumulado
ax4 = axes[1, 1]
df_pareto = df_analisis.sort_values('ahorro_total', ascending=False).copy()
df_pareto['ahorro_acumulado'] = df_pareto['ahorro_total'].cumsum()
df_pareto['ahorro_acumulado_pct'] = (df_pareto['ahorro_acumulado'] / df_pareto['ahorro_total'].sum()) * 100

ax4_twin = ax4.twinx()

# Barras de ahorro individual
top_30 = df_pareto.head(30)
x_pos = np.arange(len(top_30))
ax4.bar(x_pos, top_30['ahorro_total'], color='#16a085', alpha=0.7, edgecolor='black')

# Línea de ahorro acumulado
ax4_twin.plot(x_pos, top_30['ahorro_acumulado_pct'], color='red', marker='o', 
              linewidth=2, markersize=4, label='% Acumulado')
ax4_twin.axhline(80, color='orange', linestyle='--', linewidth=1.5, label='80% del ahorro')

ax4.set_xlabel('Productos (Top 30)', fontsize=12, fontweight='bold')
ax4.set_ylabel('Ahorro Individual (€)', fontsize=12, fontweight='bold', color='#16a085')
ax4_twin.set_ylabel('Ahorro Acumulado (%)', fontsize=12, fontweight='bold', color='red')
ax4.set_title('Diagrama de Pareto - Concentración del Ahorro', fontsize=13, fontweight='bold')
ax4.tick_params(axis='y', labelcolor='#16a085')
ax4_twin.tick_params(axis='y', labelcolor='red')
ax4_twin.legend(loc='lower right', fontsize=10)
ax4.grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('../datos/grafico_analisis_costes.png', dpi=300, bbox_inches='tight')
plt.show()

# Calcular cuántos productos representan el 80% del ahorro
productos_80 = len(df_pareto[df_pareto['ahorro_acumulado_pct'] <= 80])
print(f"El 80% del ahorro se concentra en {productos_80} productos ({(productos_80/len(df_pareto)*100):.1f}% del total)")
print(f"Ahorro total anual: {df_analisis['ahorro_total'].sum():,.2f} €")

### 10.4. Dashboard de Métricas Clave

In [ ]:
fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# Título principal
fig.suptitle('Dashboard Ejecutivo: Impacto del Modelo CatBoost en la Gestión de Inventario', 
             fontsize=18, fontweight='bold', y=0.98)

# --- Fila 1: Métricas principales ---

# KPI 1: Mejora en RMSE
ax1 = fig.add_subplot(gs[0, 0])
rmse_mejora = ((df_analisis['rmse_naive'].mean() - df_analisis['rmse_catboost'].mean()) / df_analisis['rmse_naive'].mean()) * 100
ax1.text(0.5, 0.6, f'{rmse_mejora:.1f}%', ha='center', va='center', fontsize=60, fontweight='bold', color='#2ecc71')
ax1.text(0.5, 0.25, 'Mejora en Precisión\n(Reducción RMSE)', ha='center', va='center', fontsize=14, color='#34495e')
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)
ax1.axis('off')
ax1.add_patch(plt.Rectangle((0.05, 0.05), 0.9, 0.9, fill=False, edgecolor='#2ecc71', linewidth=3))

# KPI 2: Reducción en Stock de Seguridad
ax2 = fig.add_subplot(gs[0, 1])
ss_reduccion = ((df_analisis['ss_naive'].sum() - df_analisis['ss_catboost'].sum()) / df_analisis['ss_naive'].sum()) * 100
ss_unidades = df_analisis['ss_diferencia'].sum()
ax2.text(0.5, 0.65, f'{ss_reduccion:.1f}%', ha='center', va='center', fontsize=60, fontweight='bold', color='#3498db')
ax2.text(0.5, 0.35, f'({ss_unidades:,.0f} unidades)', ha='center', va='center', fontsize=12, color='#34495e')
ax2.text(0.5, 0.2, 'Reducción en\nStock de Seguridad', ha='center', va='center', fontsize=14, color='#34495e')
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.axis('off')
ax2.add_patch(plt.Rectangle((0.05, 0.05), 0.9, 0.9, fill=False, edgecolor='#3498db', linewidth=3))

# KPI 3: Ahorro Anual
ax3 = fig.add_subplot(gs[0, 2])
ahorro_total = df_analisis['ahorro_total'].sum()
ahorro_pct = (ahorro_total / df_analisis['coste_total_naive'].sum()) * 100
ax3.text(0.5, 0.65, f'{ahorro_total/1000:,.0f}K€', ha='center', va='center', fontsize=60, fontweight='bold', color='#27ae60')
ax3.text(0.5, 0.35, f'({ahorro_pct:.1f}% reducción)', ha='center', va='center', fontsize=12, color='#34495e')
ax3.text(0.5, 0.2, 'Ahorro Anual Total', ha='center', va='center', fontsize=14, color='#34495e')
ax3.set_xlim(0, 1)
ax3.set_ylim(0, 1)
ax3.axis('off')
ax3.add_patch(plt.Rectangle((0.05, 0.05), 0.9, 0.9, fill=False, edgecolor='#27ae60', linewidth=3))

# --- Fila 2: Comparativas ---

# Comparativa Stock Medio
ax4 = fig.add_subplot(gs[1, :])
metricas = ['Stock de\nSeguridad', 'Stock\nMedio', 'Stock\nMáximo', 'Coste\nAlmacén', 'Coste\nRuptura']
naive_vals = [
    df_analisis['ss_naive'].sum(),
    df_analisis['stock_medio_naive'].sum(),
    df_analisis['stock_max_naive'].sum(),
    df_analisis['coste_almacen_naive'].sum(),
    df_analisis['coste_ruptura_naive'].sum()
]
catboost_vals = [
    df_analisis['ss_catboost'].sum(),
    df_analisis['stock_medio_catboost'].sum(),
    df_analisis['stock_max_catboost'].sum(),
    df_analisis['coste_almacen_catboost'].sum(),
    df_analisis['coste_ruptura_catboost'].sum()
]

# Normalizar para visualización (cada métrica en % respecto a Naive)
mejoras = [(n - c) / n * 100 for n, c in zip(naive_vals, catboost_vals)]

x = np.arange(len(metricas))
width = 0.35

bars1 = ax4.bar(x - width/2, [100]*len(metricas), width, label='Naive (100%)', 
                color='#e74c3c', alpha=0.7, edgecolor='black')
bars2 = ax4.bar(x + width/2, [100 - m for m in mejoras], width, label='CatBoost', 
                color='#2ecc71', alpha=0.7, edgecolor='black')

ax4.set_ylabel('Valor Relativo (%)', fontsize=13, fontweight='bold')
ax4.set_title('Comparativa Normalizada de Métricas Clave (Naive = 100%)', fontsize=15, fontweight='bold', pad=15)
ax4.set_xticks(x)
ax4.set_xticklabels(metricas, fontsize=12)
ax4.legend(fontsize=12, loc='upper right')
ax4.grid(axis='y', alpha=0.3, linestyle='--')
ax4.set_ylim(0, 120)

# Añadir etiquetas de mejora
for i, mejora in enumerate(mejoras):
    ax4.text(i, 105, f'-{mejora:.1f}%', ha='center', fontsize=11, fontweight='bold', color='#27ae60')

# --- Fila 3: Distribuciones ---

# Scatter: Ahorro vs Demanda
ax5 = fig.add_subplot(gs[2, 0])
scatter = ax5.scatter(df_analisis['demanda_media_diaria'], df_analisis['ahorro_total'],
                      c=df_analisis['ahorro_pct'], cmap='RdYlGn', s=60, alpha=0.7, 
                      edgecolors='black', linewidth=0.5)
ax5.set_xlabel('Demanda Media Diaria', fontsize=11, fontweight='bold')
ax5.set_ylabel('Ahorro Total (€)', fontsize=11, fontweight='bold')
ax5.set_title('Ahorro vs Demanda', fontsize=12, fontweight='bold')
ax5.grid(alpha=0.3, linestyle='--')
cbar = plt.colorbar(scatter, ax=ax5)
cbar.set_label('Ahorro %', fontsize=9)

# Boxplot: Comparativa RMSE
ax6 = fig.add_subplot(gs[2, 1])
bp = ax6.boxplot([df_analisis['rmse_naive'], df_analisis['rmse_catboost']], 
                  labels=['Naive', 'CatBoost'],
                  patch_artist=True,
                  boxprops=dict(facecolor='lightblue', edgecolor='black'),
                  medianprops=dict(color='red', linewidth=2),
                  whiskerprops=dict(color='black'),
                  capprops=dict(color='black'))
bp['boxes'][0].set_facecolor('#e74c3c')
bp['boxes'][1].set_facecolor('#2ecc71')
ax6.set_ylabel('RMSE (unidades)', fontsize=11, fontweight='bold')
ax6.set_title('Distribución de Errores', fontsize=12, fontweight='bold')
ax6.grid(axis='y', alpha=0.3, linestyle='--')

# Pie: Composición de costes CatBoost
ax7 = fig.add_subplot(gs[2, 2])
coste_almacen_cb = df_analisis['coste_almacen_catboost'].sum()
coste_ruptura_cb = df_analisis['coste_ruptura_catboost'].sum()
sizes = [coste_almacen_cb, coste_ruptura_cb]
labels = [f'Almacenamiento\n{coste_almacen_cb:,.0f}€', f'Ruptura\n{coste_ruptura_cb:,.0f}€']
colors = ['#3498db', '#e67e22']
explode = (0.05, 0.05)

ax7.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
        shadow=True, startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'})
ax7.set_title('Composición de Costes\n(Modelo CatBoost)', fontsize=12, fontweight='bold')

plt.savefig('../datos/grafico_dashboard_ejecutivo.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*80)
print("DASHBOARD GENERADO EXITOSAMENTE")
print("="*80)
print(f"Total de productos analizados: {len(df_analisis)}")
print(f"Mejora en precisión (RMSE): {rmse_mejora:.2f}%")
print(f"Reducción en stock de seguridad: {ss_reduccion:.2f}%")
print(f"Ahorro anual total: {ahorro_total:,.2f} €")
print(f"ROI estimado: {ahorro_pct:.2f}%")
print("="*80)